In [ ]:
!pip install -U bitsandbytes>=0.46.1
# temp fix

In [ ]:
import os
import sys
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import drive

# 1. Configurazione del percorso e Cache (Colab o Locale)
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    
    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    
    # Impostiamo HF_HOME prima di importare hf_hub o transformers
    # In questo modo i modelli vengono salvati su Google Drive e non si riscaricano.
    hf_cache_dir = "/content/drive/MyDrive/progettoLLM/hf_cache"
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    
    print(f"Ambiente Colab e Google Drive configurati. Cache modelli in: {hf_cache_dir}")
except ImportError:
    print("Ambiente locale rilevato. Verrà usata la cache Hugging Face di default.")

In [ ]:
# 2. Gestione del Token di Autenticazione
hf_token = None
try:
    from google.colab import userdata
    print("Acquisizione token da Google Secrets...")
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    print("Lettura token dal file .env locale...")
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    break
    else:
        print("Errore: File .env non trovato.")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Autenticazione Hugging Face completata.")
else:
    print("Attenzione: Token HF non trovato. Possibili errori nel download del modello.")

In [ ]:
# 3. Caricamento del Modello (Quantizzazione 4-bit)
model_id = "meta-llama/Llama-3.1-8B-Instruct" 
print(f"Inizializzazione configurazione BitsAndBytes per {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Trasferimento dei pesi del modello in VRAM...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16  # Corretto da torch_dtype
)

print("Caricamento completato con successo. Sistema pronto per l'inferenza.")

In [ ]:
prompt = "What is the capital of France?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0]))